# 📘 Tutorial 3: Multi-Fidelity and Contextual BO

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 2**, we moved from single-objective BO to **multi-objective Bayesian Optimisation**, where the optimiser no longer searches for one best scalar value, but instead learns a **Pareto frontier** of trade-offs.

In particular, we saw that once several objectives are present, the optimisation problem is no longer simply about asking:

> **which candidate gives the best objective value?**

Instead, the workflow also needs to ask:

- whether one formulation dominates another,
- which points are Pareto-optimal among the current observations,
- how the observed Pareto frontier improves over time,
- how hypervolume can summarise multi-objective progress,
- how qEHVI proposes candidates by expected hypervolume improvement,
- and how final decision-making still requires preferences after the Pareto set has been found.

That established multi-objective BO as a workflow for learning **sets of best trade-offs**, rather than one universal optimum.

In this tutorial, we take the next step:

> **what changes when the evaluation itself can happen at different fidelities, or when the best design depends on an external context?**

That is the focus of this notebook.

This is another important shift.

In earlier BO tutorials, the input variables were usually treated as design variables that the optimiser could freely choose.

But in realistic experimental optimisation, not every input dimension has the same meaning.

Some variables describe **how the experiment is evaluated**.

Some variables describe **the condition under which the experiment is judged**.

These two cases lead to two important BO settings:

- **multi-fidelity BO**,
- and **contextual BO**.

They both use augmented input spaces, but they answer different scientific questions.

---

## Multi-fidelity BO

In the first half of this notebook, we study **multi-fidelity BO** using a synthetic photocatalyst optimisation problem.

The design variable is:

- **co-catalyst loading**.

The extra variable is:

- **fidelity level**.

So the objective is written as:

$$
f(x,s),
$$

where:

- $x$ is the co-catalyst loading,
- $s$ is the fidelity level.

The high-fidelity target is:

$$
f(x,s=1).
$$

This represents the full, accurate, expensive evaluation that we ultimately care about.

Lower-fidelity evaluations are cheaper, but biased approximations of this target.

So the optimiser is no longer asking only:

> **which co-catalyst loading should we evaluate next?**

It is also asking:

> **at what fidelity should we evaluate it?**

This is the central question of multi-fidelity BO.

Low-fidelity experiments can help guide the search because they are cheaper, but they are not the final target.

The final recommendation must still be judged by high-fidelity performance at:

$$
s = 1.
$$

---

## Contextual BO

In the second half of this notebook, we study **contextual BO**.

Here, the extra variable is no longer a fidelity level.

Instead, it is a **light-intensity context**.

The objective becomes:

$$
f(x,c),
$$

where:

- $x$ is the co-catalyst loading,
- $c$ is the light-intensity context.

The key difference is that $c$ is not a cheaper or more expensive version of the experiment.

It is the condition under which the design is judged.

So contextual BO does not ask:

> **which design is best overall?**

It asks:

> **given the current context, which design should we choose?**

This means the best co-catalyst loading can change as the context changes.

There may not be one universally best loading.

Instead, the optimiser needs to learn a context-dependent recommendation rule:

$$
c \mapsto x^*(c).
$$

That is the central idea of contextual BO.

---

## Why these topics belong together

Multi-fidelity BO and contextual BO are grouped together because both require us to be more careful about the meaning of the input variables.

In the simplest BO setting, every input dimension is usually treated as a design variable that the optimiser can freely choose.

Here, that assumption no longer holds.

In the multi-fidelity setting, the extra variable $s$ describes **how the evaluation is performed**.

A lower-fidelity evaluation is cheaper, but it is only an imperfect proxy for the high-fidelity target. So $s$ changes the reliability and cost of the information we obtain.

In the contextual setting, the extra variable $c$ describes **the condition under which the design is judged**.

The context is not a cheaper approximation and is not usually chosen by the optimiser. Instead, it is observed or supplied, and the optimiser chooses the best design conditional on that context.

So both parts of the notebook use augmented inputs:

$$
(x,s) \quad \text{or} \quad (x,c),
$$

but the interpretation is different.

Fidelity changes the **evaluation process**.

Context changes the **meaning of a good design**.

This distinction is the conceptual link between the two halves of the tutorial.

---

## What this notebook does

To make these ideas concrete, the notebook is split into two connected parts.

The first part defines a synthetic **photocatalytic activity** target and introduces lower-fidelity approximations.

It then compares:

- a **high-fidelity-only BO baseline**,
- and a **multi-fidelity acquisition-per-cost BO workflow**.

This lets us study how cheap biased evaluations can help guide the search toward a high-fidelity optimum under a limited budget.

The second part defines a **contextual photocatalysis objective**, where the best co-catalyst loading shifts with light-intensity context.

It then runs contextual BO by receiving a sequence of context values and choosing a co-catalyst loading conditional on each one.

This lets us study how BO can learn recommendations that depend on external operating conditions.

---

**This tutorial is designed to shift perspective**
- from *“BO chooses the next design point”*
- to *“BO may also need to choose how accurately to evaluate a design, or adapt the design choice to the current context.”*

---

**The emphasis is on developing intuition for**
- why low-fidelity evaluations can be useful but biased,
- why high-fidelity performance remains the final target in multi-fidelity BO,
- why acquisition-per-cost can support budget-aware fidelity selection,
- why cheap evaluations can be over-selected if the cost penalty is too strong,
- why contextual BO does not search for one universal optimum,
- why the best design can change with context,
- and why coverage of the context space affects contextual BO performance.

---

**Key ideas explored include**
- defining a synthetic high-fidelity photocatalytic activity target,
- constructing lower-fidelity biased approximations,
- assigning evaluation cost as a function of fidelity,
- comparing high-fidelity-only BO with multi-fidelity BO under a fixed budget,
- inspecting which fidelity levels are selected by the workflow,
- distinguishing final high-fidelity recommendation quality from posterior curve quality,
- defining a contextual photocatalysis objective,
- visualising how the optimum shifts with light-intensity context,
- running contextual BO with fixed observed contexts,
- and comparing model recommendations against the true contextual activity curves.

---

This tutorial serves as the bridge from:

- **multi-objective BO** in Tutorial 2,
- to **even richer BO settings where fidelity, context, structure, and practical workflow design matter**.

In other words:

- **Tutorial 2** showed how BO changes when there are several competing objectives and no single universal best point,
- and **Tutorial 3** now asks what happens when the evaluation process itself has different fidelities, or when the best design depends on the context in which it is tested.

---

**Recommended prerequisites**
- Completion of **Tutorials 1–2 of Part 6**
- Familiarity with Gaussian Process surrogates and acquisition functions in BoTorch
- Familiarity with the standard sequential BO loop
- Basic understanding of UCB-style acquisition functions
- Basic awareness of multi-objective BO, Pareto frontiers, and hypervolume
- Some intuition for experimental cost, proxy measurements, and context-dependent performance

---

**Author**: Angze Li

**Last updated**: 2026-04-25

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.utils.sampling import draw_sobol_samples
from gpytorch.mlls import ExactMarginalLogLikelihood

warnings.filterwarnings("ignore")

torch.set_default_dtype(torch.double)

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")

## 1. Defining the high-fidelity photocatalytic activity target

This cell sets up the **high-fidelity target function** for the multi-fidelity part of the notebook.

The scientific picture is the following.

We imagine that we are optimising a photocatalyst using one controllable design variable:

- **co-catalyst loading**.

Here, the variable `x` is scaled to the interval $[0,1]$, so it should be interpreted as a **normalised co-catalyst loading level** rather than a literal experimental unit.

The function `high_fidelity_activity_from_x(x)` then defines the **true high-fidelity photocatalytic activity** associated with each loading.

That high-fidelity activity is the quantity we ultimately care about.

---

### What “fidelity” means here

The word **fidelity** refers to the **quality, realism, or accuracy of an evaluation**.

A high-fidelity evaluation is one that is closer to the true experimental target, but it is usually more expensive, slower, or harder to obtain.

A low-fidelity evaluation is cheaper or faster, but it is only an approximation of the true target.

In this tutorial, the high-fidelity function represents something like:

- a full-duration photocatalytic activity test,
- a more careful or realistic experimental measurement,
- or a gold-standard evaluation that we trust most.

Later in the notebook, we will introduce lower-fidelity evaluations that are cheaper but biased approximations of this target.

So the key distinction is:

- **high fidelity** = the target we ultimately want to optimise,
- **low fidelity** = a cheaper proxy that may still contain useful information.

---

### Why we define the high-fidelity target first

Before introducing lower-fidelity approximations, we need a clear definition of the target they are approximating.

That is why this cell only defines the high-fidelity activity.

Conceptually, multi-fidelity BO is not trying to optimise a cheap proxy for its own sake.

It is trying to use cheaper proxy evaluations to learn about the true target more efficiently.

So this high-fidelity function is the anchor of the whole tutorial.

---

### What the code does

The code first sets:

- the random seed for reproducibility,
- the 1D search interval $[0,1]$ for co-catalyst loading,
- and a dense grid `grid_x` for visualisation.

The function `high_fidelity_activity_from_x(x)` then defines a smooth synthetic activity landscape with multiple local structure but one true global optimum.

The helper `high_fidelity_objective(X)` wraps this into a BO-friendly tensor shape.

Finally, the code evaluates the function on the dense grid and identifies:

- `true_best_x`: the true best co-catalyst loading on the grid,
- `true_best_y`: the corresponding high-fidelity activity.

These values act as a reference for later comparisons.

---

### Why this is useful for the tutorial

Because the objective is synthetic, we know the true high-fidelity optimum exactly on the dense grid.

That is extremely helpful for teaching.

It means we can later compare:

- what the BO workflow recommends,
- how close that recommendation is to the true optimum,
- whether low-fidelity evaluations help identify the correct region,
- and how budget-aware multi-fidelity BO behaves compared with high-fidelity-only BO.

In a real experimental campaign, we would not know the full high-fidelity function in advance.

Here, we do know it, so we can use it as a ground-truth diagnostic.

---

### Key takeaway

This cell defines the **true high-fidelity photocatalytic activity target** as a function of co-catalyst loading.

This is the quantity we ultimately want to optimise.

Later, lower-fidelity evaluations will be introduced as cheaper but biased approximations of this target.

So the main conceptual distinction is:

> **high fidelity tells us what we really care about, while low fidelity gives cheaper but less reliable information about it.**

In [ ]:
seed = 21
set_seed(seed)

bounds_x = torch.tensor([[0.0], [1.0]], dtype=torch.double)

grid_x = torch.linspace(0.0, 1.0, 500).unsqueeze(-1)

TRUE_COLOR = "#00E5FF"

def high_fidelity_activity_from_x(x):
    return (
        0.35
        + 0.95 * torch.exp(-0.5 * ((x - 0.68) / 0.14) ** 2)
        + 0.38 * torch.exp(-0.5 * ((x - 0.27) / 0.10) ** 2)
        + 0.04 * torch.sin(18.0 * x + 0.3)
        - 0.08 * (x - 0.55) ** 2
    )


def high_fidelity_objective(X):
    if X.ndim == 1:
        X = X.unsqueeze(-1)

    x = X[..., 0]
    y = high_fidelity_activity_from_x(x)

    return y.unsqueeze(-1)


grid_y_high = high_fidelity_objective(grid_x)

true_best_idx = torch.argmax(grid_y_high)
true_best_x = grid_x[true_best_idx]
true_best_y = grid_y_high[true_best_idx]

print("True high-fidelity optimum x:", float(true_best_x))
print("True high-fidelity activity:", float(true_best_y))

## 2. Defining lower-fidelity approximations of the photocatalytic activity

This cell extends the high-fidelity photocatalytic target into a **multi-fidelity objective**.

Instead of evaluating only the full high-fidelity experiment, we now allow the same co-catalyst loading to be evaluated at different **fidelity levels**:

- $s = 1.0$ represents the full high-fidelity evaluation,
- lower values of $s$ represent cheaper but less reliable proxy evaluations.

So the objective now depends on two quantities:

$$
f(x, s),
$$

where $x$ is the co-catalyst loading and $s$ is the fidelity level.

---

### What the multi-fidelity objective represents

The function `multi_fidelity_objective(X)` treats each input row as a pair:

```python
X[..., 0] = co-catalyst loading x
X[..., 1] = fidelity level s
```

The high-fidelity target is still computed first:

```python
high = high_fidelity_activity_from_x(x)
```

This represents the activity value we would obtain from the full, accurate photocatalytic test.

The lower-fidelity objective is then constructed by modifying this high-fidelity target using two terms:

1. a structured fidelity-dependent bias,
2. a low-fidelity penalty.

This makes the lower-fidelity curves informative, but not identical to the true target.

---

### Structured fidelity-dependent bias

The term

```python
fidelity_bias = (
    0.22 * torch.exp(-0.5 * ((x - 0.38) / 0.17) ** 2)
    - 0.18 * torch.exp(-0.5 * ((x - 0.78) / 0.13) ** 2)
    + 0.06 * torch.sin(9.0 * x + 0.5)
)
```

creates a bias pattern across the co-catalyst loading axis.

This means lower-fidelity evaluations are not just randomly wrong. They are **systematically distorted** relative to the high-fidelity target.

That is realistic for many proxy experiments.

For example, a short screening test may overestimate activity in one loading region and underestimate activity in another.

The bias is scaled by:

$$
1 - s.
$$

So the final bias contribution is:

$$
(1-s)\,\text{fidelity\_bias}.
$$

This gives the desired behaviour:

- when $s = 1.0$, the bias disappears,
- when $s = 0.5$, the bias is partly present,
- when $s = 0.2$, the bias is much stronger.

So lower fidelity means stronger deviation from the high-fidelity target.

---

### How the low-fidelity penalty is achieved

The low-fidelity penalty is defined as:

```python
low_fidelity_penalty = 0.05 * (1.0 - s)
```

This term is then subtracted in the final objective:

```python
y = high + (1.0 - s) * fidelity_bias - low_fidelity_penalty
```

Mathematically, this means:

$$
f(x,s)
=
f_{\text{high}}(x)
+
(1-s)b(x)
-
0.05(1-s),
$$

where $b(x)$ is the structured fidelity bias.

The penalty depends on $1-s$, so it becomes larger as fidelity decreases:

- if $s = 1.0$, then $0.05(1-s) = 0$,
- if $s = 0.5$, then $0.05(1-s) = 0.025$,
- if $s = 0.2$, then $0.05(1-s) = 0.040$.

So high fidelity has no penalty, while lower-fidelity evaluations are pushed slightly downward.

This makes the low-fidelity measurement not only biased, but also somewhat pessimistic compared with the full target.

---

### Why include a penalty at all?

The penalty helps make the fidelity hierarchy clearer.

Without it, a low-fidelity curve might sometimes look just as good as, or even better than, the high-fidelity target over broad regions.

That would make the tutorial harder to interpret, because low fidelity would look like an alternative objective rather than a cheaper approximation.

By subtracting a penalty that grows as fidelity decreases, the code reinforces the intended meaning:

> lower-fidelity evaluations may be useful, but they should not be treated as equally reliable measurements of the final target.

This is exactly the point of multi-fidelity BO.

We want cheap evaluations to help guide the search, but the final recommendation should still be judged by the high-fidelity objective at $s = 1$.

---

### Fidelity levels used in this tutorial

The line

```python
fidelity_levels = torch.tensor([0.2, 0.5, 1.0], dtype=torch.double)
```

defines three available evaluation fidelities:

- $s = 0.2$: low fidelity,
- $s = 0.5$: medium fidelity,
- $s = 1.0$: high fidelity.

Later, the BO workflow will be allowed to choose between these levels.

This lets us study the key multi-fidelity question:

> should we spend budget on a cheap approximate evaluation, or on a more expensive high-fidelity one?

---

### Key takeaway

This cell defines a synthetic multi-fidelity photocatalytic activity function.

The high-fidelity target remains the quantity we ultimately care about, while lower-fidelity evaluations are created by adding a structured bias and subtracting a penalty that grows as fidelity decreases.

So the main idea is:

> low fidelity is cheaper and informative, but biased and penalised; high fidelity is expensive but represents the true optimisation target.

In [ ]:
def multi_fidelity_objective(X):
    if X.ndim == 1:
        X = X.unsqueeze(0)

    x = X[..., 0]
    s = X[..., 1]

    high = high_fidelity_activity_from_x(x)

    fidelity_bias = (
        0.22 * torch.exp(-0.5 * ((x - 0.38) / 0.17) ** 2)
        - 0.18 * torch.exp(-0.5 * ((x - 0.78) / 0.13) ** 2)
        + 0.06 * torch.sin(9.0 * x + 0.5)
    )

    low_fidelity_penalty = 0.05 * (1.0 - s)

    y = high + (1.0 - s) * fidelity_bias - low_fidelity_penalty

    return y.unsqueeze(-1)


fidelity_levels = torch.tensor([0.2, 0.5, 1.0], dtype=torch.double)

print("Fidelity levels:", fidelity_levels.tolist())

## 3. Visualising fidelity: short biased screens vs full experiments

This cell visualises the synthetic photocatalytic activity curve at several different fidelity levels.

The purpose is to show that lower-fidelity evaluations are not identical copies of the high-fidelity target.

They are cheaper approximations, but they are also biased.

In this example, the three curves correspond to:

- $s = 0.2$: low fidelity,
- $s = 0.5$: medium fidelity,
- $s = 1.0$: high fidelity.

The high-fidelity curve, $s = 1.0$, is the true target we ultimately care about optimising.

---

### What the plot shows

For each fidelity level, the code constructs an augmented input tensor:

```python
X_aug = torch.cat(
    [
        grid_x,
        torch.full_like(grid_x, float(s)),
    ],
    dim=-1,
)
```

Each row of `X_aug` contains both:

```python
[x, s]
```

where:

- $x$ is the co-catalyst loading,
- $s$ is the fidelity level.

The function `multi_fidelity_objective(X_aug)` then evaluates the activity curve at that fidelity.

So this figure is not showing three different catalyst systems.

It is showing the same underlying optimisation problem observed through three different evaluation fidelities.

---

### Lower fidelity shifts both the activity values and the apparent landscape

The important point is that lower fidelity changes both the **vertical values** and the **shape of the curve**.

In other words, lower fidelity shifts the observed behaviour in both:

- the **$y$ direction**, because the measured activity values are biased upward or downward,
- and the **$x$ direction**, because the locations of peaks and shoulders can appear slightly shifted or distorted.

This is why low-fidelity data should not be treated as if they were exact high-fidelity observations.

For example, the low-fidelity curve may make some loading regions look more attractive than they truly are at high fidelity, while making other regions look less attractive.

That is exactly the kind of situation multi-fidelity BO is designed for.

---

### Why the lower-fidelity curves differ

The lower-fidelity curves differ because the objective was defined as:

$$
f(x,s)
=
f_{\text{high}}(x)
+
(1-s)b(x)
-
0.05(1-s),
$$

where $b(x)$ is the structured fidelity bias.

The term $(1-s)b(x)$ creates a fidelity-dependent distortion of the curve.

The term $0.05(1-s)$ creates a downward penalty for lower fidelity.

Together, these terms make lower-fidelity evaluations informative but imperfect.

When $s = 1.0$, both the bias scaling and penalty vanish, so the curve becomes the high-fidelity target.

When $s < 1.0$, the curve is shifted and distorted relative to the high-fidelity target.

---

### What the dashed line means

The dashed vertical line marks the true optimum of the high-fidelity target.

This is the value of $x$ that maximises the $s = 1.0$ curve on the dense grid.

It is important that this line is defined by the high-fidelity objective, not by the low-fidelity curves.

So even if a low-fidelity curve appears to have a slightly different shape or a slightly different apparent optimum, the final target remains:

$$
f(x, s=1).
$$

That is the quantity the BO workflow should ultimately optimise.

---

### Key takeaway

This figure shows why multi-fidelity optimisation is useful but also delicate.

Lower-fidelity evaluations can reveal useful information about the high-fidelity activity landscape, but they can also shift both the apparent activity values and the apparent shape of the response curve.

So the main idea is:

> low-fidelity evaluations are useful proxies, not replacements for the high-fidelity target.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for s in fidelity_levels:
    X_aug = torch.cat(
        [
            grid_x,
            torch.full_like(grid_x, float(s)),
        ],
        dim=-1,
    )

    y_s = multi_fidelity_objective(X_aug)

    ax.plot(
        grid_x.view(-1),
        y_s.view(-1),
        lw=2.5,
        label=f"fidelity s = {float(s):.1f}",
    )

ax.axvline(float(true_best_x), linestyle="--", lw=2.0, color=TRUE_COLOR, label="True high-fidelity optimum")

ax.set_title("Photocatalytic activity at different fidelities", fontsize=18, fontweight="bold")
ax.set_xlabel("Co-catalyst loading", fontsize=22, fontweight="bold")
ax.set_ylabel("Activity", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 4. Defining evaluation cost by fidelity

This cell defines the **cost model** for the multi-fidelity part of the notebook.

Up to this point, different fidelity levels have only affected the activity value returned by the synthetic objective.

Now we add the other essential part of a multi-fidelity workflow:

> lower-fidelity evaluations should be cheaper than high-fidelity evaluations.

That is what makes the optimisation problem interesting.

If all fidelity levels had the same cost, there would be little reason to use lower-fidelity approximations. We could simply evaluate everything at the high-fidelity target.

---

### The cost model

The cost of an evaluation is defined as:

$$
\mathrm{cost}(s) = 0.15 + 0.85s^2.
$$

This is implemented by:

```python
def fidelity_cost_from_s(s):
    return 0.15 + 0.85 * s ** 2
```

This means that cost increases nonlinearly with fidelity.

So:

- low fidelity is cheap,
- medium fidelity is moderately expensive,
- high fidelity is the most expensive.

For the fidelity levels used in this tutorial:

- $s = 0.2$ has low cost,
- $s = 0.5$ has intermediate cost,
- $s = 1.0$ has cost $1.0$.

The exact numbers are synthetic, but the structure is realistic: more accurate or more complete experiments usually cost more time, materials, or instrument resources.

---

### Why the functions are separated

The two functions are separated because they operate at different levels of abstraction.

`fidelity_cost_from_s(...)` works directly on fidelity values:

$$
s \mapsto \mathrm{cost}(s).
$$

`fidelity_cost_from_X(...)` works on full BO candidate inputs:

$$
[x,s] \mapsto \mathrm{cost}(s).
$$

This separation avoids repeating the same formula in multiple places.

It also reduces the chance of inconsistency.

If we later want to change the cost model, we only need to modify `fidelity_cost_from_s(...)`, and every part of the notebook that calls `fidelity_cost_from_X(...)` will automatically use the updated cost model.

---

### What the table shows

The dataframe `cost_df` displays the cost assigned to each allowed fidelity level.

This is a useful sanity check before running BO.

It confirms that:

- $s = 0.2$ is cheap,
- $s = 0.5$ is more expensive,
- $s = 1.0$ is the most expensive high-fidelity evaluation.

This table is also useful because the next BO workflow will explicitly trade off:

- expected usefulness of a candidate,
- against its evaluation cost.

---

### Key takeaway

This cell defines the cost structure that makes the tutorial genuinely multi-fidelity.

The lower-fidelity evaluations are not only biased approximations of the high-fidelity target; they are also cheaper to run.

The two cost functions are separated for clarity:

- `fidelity_cost_from_s(...)` defines the mathematical cost model from fidelity alone,
- `fidelity_cost_from_X(...)` applies that same model to full BO candidate inputs of the form `[x, s]`.

So the main idea is:

> multi-fidelity BO is not only about different evaluation qualities, but also about deciding how to spend limited budget across cheap approximate tests and expensive high-fidelity experiments.

In [ ]:
def fidelity_cost_from_s(s):
    return 0.15 + 0.85 * s ** 2

def fidelity_cost_from_X(X):
    s = X[..., 1]
    return fidelity_cost_from_s(s)


cost_df = pd.DataFrame({
    "fidelity": fidelity_levels.detach().cpu().numpy(),
    "cost": fidelity_cost_from_s(fidelity_levels).detach().cpu().numpy(),
})

display(cost_df)

## 5. GP helpers, UCB candidate selection, and high-fidelity recommendation

This cell defines three helper functions used in the BO loops:

1. `fit_augmented_gp(...)` fits the GP surrogate,
2. `select_ucb_candidate(...)` selects the next candidate,
3. `recommend_high_fidelity_from_model(...)` gives the current best high-fidelity recommendation.

---

### Fitting a GP on augmented inputs

The function `fit_augmented_gp(...)` fits a `SingleTaskGP` to the current data.

In the multi-fidelity section, each input has the form:

```python
[x, s]
```

where $x$ is co-catalyst loading and $s$ is fidelity.

So the GP models:

$$
f(x,s).
$$

Later, the same helper is also used for contextual BO, where the input becomes:

```python
[x, c]
```

where $c$ is the light-intensity context.

The input is normalised and the output is standardised to make GP fitting more stable.

---

### Why UCB is used instead of LogEI

The candidate-selection helper uses Upper Confidence Bound:

$$
\mathrm{UCB}(x) = \mu(x) + \beta\sigma(x).
$$

This favours points that are either predicted to be good, uncertain, or both.

That is useful here because the multi-fidelity candidate pool contains points at different fidelities:

$$
(x,s).
$$

A naive LogEI over this full pool would ask which $(x,s)$ improves the best observed value. But that is not quite the right question, because low-fidelity values are biased proxies rather than the final target.

The real target is specifically:

$$
f(x,s=1).
$$

So UCB is used as a simple information-seeking rule instead of treating low-fidelity improvement as equivalent to high-fidelity improvement.

---

### Cost-aware UCB

The optional `utility_denominator` lets us divide UCB by evaluation cost:

$$
\mathrm{utility}(x,s)
=
\frac{\mathrm{UCB}(x,s)}{\mathrm{cost}(s)}.
$$

This is the simple acquisition-per-cost heuristic used later.

It asks:

> which candidate gives the most promising or informative opportunity per unit cost?

This is not a fully principled multi-fidelity acquisition function, but it is transparent and useful for a first tutorial.

---

### High-fidelity recommendation

The function `recommend_high_fidelity_from_model(...)` is important because the final recommendation should always be made at the target fidelity:

$$
s = 1.
$$

It constructs a grid of candidates:

```python
[x, 1.0]
```

then chooses the $x$ with the highest posterior mean on that high-fidelity slice.

So the recommendation is not:

> which low-fidelity observation looked best?

It is:

> which co-catalyst loading does the model currently believe is best at high fidelity?

The function also returns the true high-fidelity value at that recommendation. In a real experiment, this would be unknown, but in this synthetic tutorial it is useful for diagnostics.

---

### Key takeaway

This cell defines the basic modelling and decision helpers for the multi-fidelity workflow.

Low-fidelity evaluations can help the model learn, but the final recommendation is always judged on the high-fidelity target $f(x,s=1)$.

In [ ]:
def fit_augmented_gp(train_X, train_Y):
    gp = SingleTaskGP(
        train_X=train_X,
        train_Y=train_Y,
        input_transform=Normalize(d=train_X.shape[-1]),
        outcome_transform=Standardize(m=1),
    )

    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_mll(mll)
    gp.eval()

    return gp


def select_ucb_candidate(model, candidate_X, beta=1.5, utility_denominator=None):
    with torch.no_grad():
        posterior = model.posterior(candidate_X)
        mean = posterior.mean.squeeze(-1)
        std = posterior.variance.sqrt().squeeze(-1)

    ucb = mean + beta * std

    if utility_denominator is None:
        utility = ucb
    else:
        utility = ucb / utility_denominator.clamp_min(1e-12)

    best_idx = torch.argmax(utility)

    return candidate_X[best_idx:best_idx + 1], {
        "posterior_mean": float(mean[best_idx]),
        "posterior_std": float(std[best_idx]),
        "ucb": float(ucb[best_idx]),
        "utility": float(utility[best_idx]),
    }


def recommend_high_fidelity_from_model(model, grid_x):
    target_X = torch.cat(
        [
            grid_x,
            torch.ones_like(grid_x),
        ],
        dim=-1,
    )

    with torch.no_grad():
        posterior = model.posterior(target_X)
        mean = posterior.mean.squeeze(-1)
        std = posterior.variance.sqrt().squeeze(-1)

    best_idx = torch.argmax(mean)

    x_rec = grid_x[best_idx:best_idx + 1]
    y_rec_true = high_fidelity_objective(x_rec)

    return x_rec, y_rec_true, mean[best_idx], std[best_idx]

## 6. Initial designs and budget normalisation

This cell constructs the initial datasets for the two BO workflows that will be compared next:

- a **high-fidelity-only** workflow,
- and a **multi-fidelity** workflow.

Both workflows start from the same five co-catalyst loading values, but they evaluate those loadings at different fidelity levels.

This is important because the comparison should not be driven by different initial $x$ locations.

Instead, the difference should come from the evaluation strategy:

> high-fidelity-only BO spends more budget per initial point, while multi-fidelity BO starts with a cheaper mixture of low-, medium-, and high-fidelity evaluations.

---

### Same initial co-catalyst loadings

The initial co-catalyst loadings are generated using Sobol sampling:

```python
init_x = draw_sobol_samples(
    bounds=bounds_x,
    n=1,
    q=n_init,
).squeeze(0)
```

This gives `n_init = 5` initial loading values in the interval $[0,1]$.

These same loading values are reused for both workflows.

So both methods begin with the same initial coverage of the co-catalyst loading axis.

---

### High-fidelity-only initial design

For the high-fidelity-only workflow, every initial point is evaluated at:

$$
s = 1.
$$

This is created by concatenating the initial loading values with a column of ones:

```python
train_X_hf_init = torch.cat(
    [
        init_x,
        torch.ones_like(init_x),
    ],
    dim=-1,
)
```

So every initial input has the form:

```python
[x, 1.0]
```

This means all five initial evaluations are full high-fidelity experiments.

Since the cost model from the previous section was:

$$
\mathrm{cost}(s) = 0.15 + 0.85s^2,
$$

a high-fidelity evaluation costs:

$$
\mathrm{cost}(1.0) = 1.0.
$$

Therefore, five high-fidelity initial evaluations cost:

$$
5 \times 1.0 = 5.0.
$$

---

### Multi-fidelity initial design

For the multi-fidelity workflow, the same five loading values are paired with a mixture of fidelity levels:

```python
init_s = torch.tensor([[0.2], [0.5], [1.0], [0.2], [0.5]], dtype=torch.double)
```

So the initial multi-fidelity dataset contains:

- two low-fidelity evaluations at $s=0.2$,
- two medium-fidelity evaluations at $s=0.5$,
- one high-fidelity evaluation at $s=1.0$.

This gives the model some information across fidelities while spending much less budget than evaluating all five points at $s=1$.

---

### Linking budget to `fidelity_cost_from_X(...)`

The initial costs are computed using:

```python
initial_hf_cost = float(fidelity_cost_from_X(train_X_hf_init).sum())
initial_mf_cost = float(fidelity_cost_from_X(train_X_mf_init).sum())
```

This links directly to the cost functions defined earlier.

Recall that each BO input is an augmented vector:

```python
[x, s]
```

The helper `fidelity_cost_from_X(...)` extracts the fidelity column:

```python
s = X[..., 1]
```

and applies:

$$
\mathrm{cost}(s) = 0.15 + 0.85s^2.
$$

So the budget calculation is not a separate bookkeeping trick.

It is directly tied to the fidelity level of each evaluation.

This is the key idea:

> the cost of an experiment is determined by its fidelity.

---

### Why this matters

This is the first place where the difference between high-fidelity-only BO and multi-fidelity BO becomes concrete.

The high-fidelity-only workflow begins with more direct target information, because all its initial observations are at $s=1$.

But it spends a large fraction of the total budget immediately.

The multi-fidelity workflow begins with less direct information about the target, because most of its initial observations are biased low- or medium-fidelity evaluations.

But it preserves much more budget for later sequential decisions.

So the trade-off is:

> high-fidelity-only BO starts with more accurate information, while multi-fidelity BO starts with cheaper information and more remaining budget.

---

### Key takeaway

This cell creates matched initial datasets for the high-fidelity-only and multi-fidelity workflows.

Both use the same initial co-catalyst loading values, but they differ in fidelity assignment.

The function `fidelity_cost_from_X(...)` converts those fidelity choices into actual budget consumption.

This shows the central budget logic of multi-fidelity BO:

> lower-fidelity evaluations are not only different in accuracy; they also change how much budget remains for future optimisation.

In [ ]:
set_seed(seed)

n_init = 5
budget_total = 8.0

init_x = draw_sobol_samples(
    bounds=bounds_x,
    n=1,
    q=n_init,
).squeeze(0)

train_X_hf_init = torch.cat(
    [
        init_x,
        torch.ones_like(init_x),
    ],
    dim=-1,
)

train_Y_hf_init = multi_fidelity_objective(train_X_hf_init)

init_s = torch.tensor([[0.2], [0.5], [1.0], [0.2], [0.5]], dtype=torch.double)

train_X_mf_init = torch.cat([init_x, init_s], dim=-1)
train_Y_mf_init = multi_fidelity_objective(train_X_mf_init)

initial_hf_cost = float(fidelity_cost_from_X(train_X_hf_init).sum())
initial_mf_cost = float(fidelity_cost_from_X(train_X_mf_init).sum())

high_fidelity_unit_cost = float(fidelity_cost_from_s(torch.tensor(1.0)))

print("Total budget:", budget_total)
print()

print("Initial high-fidelity cost:", initial_hf_cost)
print("Initial high-fidelity cost as % of budget:", 100 * initial_hf_cost / budget_total)
print()

print("Initial multi-fidelity cost:", initial_mf_cost)
print("Initial multi-fidelity cost as % of budget:", 100 * initial_mf_cost / budget_total)

## 7. High-fidelity-only BO baseline

This cell defines and runs the **high-fidelity-only BO baseline**.

The purpose of this baseline is simple:

> what happens if we ignore lower-fidelity evaluations and spend the budget only on full high-fidelity experiments?

This gives us a reference point for judging whether the later multi-fidelity workflow is actually useful.

---

### What this baseline does

The function `run_high_fidelity_only_bo(...)` starts from the high-fidelity initial dataset created earlier.

Every training input has the form:

```python
[x, 1.0]
```

where:

- $x$ is the co-catalyst loading,
- $s = 1.0$ means the full high-fidelity evaluation.

So this workflow never chooses low- or medium-fidelity evaluations.

It only evaluates the target fidelity:

$$
f(x, s=1).
$$

This makes the method conceptually clean, but expensive.

---

### How the BO loop is constructed

At each BO step, the loop follows the standard sequential BO structure:

1. fit a GP surrogate to the current data,
2. make the current high-fidelity recommendation using `recommend_high_fidelity_from_model(...)`,
3. record the current state in `history`,
4. check whether another high-fidelity evaluation fits within the remaining budget,
5. build a candidate grid with all candidates fixed at $s=1$,
6. select the next candidate using UCB,
7. evaluate the candidate,
8. append it to the training data.

The candidate grid is constructed as:

```python
candidate_X = torch.cat(
    [
        grid_x,
        torch.ones_like(grid_x),
    ],
    dim=-1,
)
```

This means every candidate lies on the high-fidelity slice.

So although the model input is still two-dimensional, the optimiser only searches over $x$ while keeping $s=1$ fixed.

---

### Why the budget check matters

Each high-fidelity evaluation costs:

$$
\mathrm{cost}(1.0)=1.0.
$$

Before adding a new candidate, the code checks:

```python
if spent + next_cost > budget_total:
    break
```

This prevents the loop from exceeding the total budget.

Since the high-fidelity initial design already costs $5.0$ and the total budget is $8.0$, this baseline can only afford a few additional high-fidelity evaluations.

That is the main limitation of high-fidelity-only BO:

> it receives accurate information, but each new observation is expensive.

---

### What is stored in `history`

The `history` list records the optimisation state at every step.

For each step, it stores quantities such as:

- cumulative cost,
- number of observations,
- current recommended co-catalyst loading,
- true high-fidelity activity at that recommendation,
- posterior mean and uncertainty at the recommendation,
- and the method name.

This is useful because later plots compare workflows by **cumulative cost**, not only by BO iteration.

That matters in multi-fidelity BO because different evaluations have different costs.

---

### Why this baseline is necessary

The high-fidelity-only workflow is not meant to be the most budget-efficient strategy.

It is a baseline.

It tells us how well BO performs when every evaluation is accurate but expensive.

The later multi-fidelity workflow will be judged against this baseline.

If multi-fidelity BO can reach a similar high-fidelity recommendation while spending less budget or collecting more information, then the low-fidelity evaluations are useful.

---

### Key takeaway

This cell runs the baseline BO strategy where every evaluation is performed at $s=1$.

It gives accurate high-fidelity information, but it consumes budget quickly.

So it provides the comparison point for the next workflow, where BO is allowed to trade off cheap low-fidelity evaluations against expensive high-fidelity confirmation.

In [ ]:
def run_high_fidelity_only_bo(
    train_X_init,
    train_Y_init,
    budget_total,
    max_steps=20,
    beta=1.5,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()

    spent = float(fidelity_cost_from_X(train_X).sum())
    history = []

    for step in range(max_steps + 1):
        model = fit_augmented_gp(train_X, train_Y)
        x_rec, y_rec_true, mean_rec, std_rec = recommend_high_fidelity_from_model(model, grid_x)

        history.append({
            "step": step,
            "cumulative_cost": spent,
            "n_observations": train_X.shape[0],
            "recommended_x": float(x_rec),
            "recommended_true_high_fidelity": float(y_rec_true),
            "recommendation_mean": float(mean_rec),
            "recommendation_std": float(std_rec),
            "last_fidelity": None if step == 0 else 1.0,
            "method": "High-fidelity only",
        })

        if step == max_steps:
            break

        next_cost = float(fidelity_cost_from_s(torch.tensor(1.0)))

        if spent + next_cost > budget_total:
            break

        candidate_X = torch.cat(
            [
                grid_x,
                torch.ones_like(grid_x),
            ],
            dim=-1,
        )

        candidate, info = select_ucb_candidate(
            model=model,
            candidate_X=candidate_X,
            beta=beta,
        )

        Y_new = multi_fidelity_objective(candidate)

        train_X = torch.cat([train_X, candidate], dim=0)
        train_Y = torch.cat([train_Y, Y_new], dim=0)

        spent += next_cost

    final_model = fit_augmented_gp(train_X, train_Y)

    return {
        "history": history,
        "train_X_final": train_X,
        "train_Y_final": train_Y,
        "model_final": final_model,
    }


set_seed(seed)

hf_result = run_high_fidelity_only_bo(
    train_X_init=train_X_hf_init,
    train_Y_init=train_Y_hf_init,
    budget_total=budget_total,
    max_steps=20,
    beta=1.5,
)

print("High-fidelity-only observations:", hf_result["train_X_final"].shape[0])
print("Final cost:", hf_result["history"][-1]["cumulative_cost"])
print("Final recommended x:", hf_result["history"][-1]["recommended_x"])
print("Final true high-fidelity activity:", hf_result["history"][-1]["recommended_true_high_fidelity"])

## 8. Simple multi-fidelity acquisition-per-cost BO

This cell defines and runs the **multi-fidelity BO workflow**.

Unlike the high-fidelity-only baseline, this workflow is allowed to choose both:

- the co-catalyst loading $x$,
- and the fidelity level $s$.

So each candidate has the form:

```python
[x, s]
```

where $s$ can be low, medium, or high fidelity.

This is the key difference from the previous baseline.

The optimiser is no longer asking only:

> which co-catalyst loading should I evaluate next?

It is now asking:

> which co-catalyst loading should I evaluate, and at what fidelity?

---

### Building the multi-fidelity candidate pool

The function `make_multi_fidelity_candidate_pool(...)` creates a grid over both co-catalyst loading and fidelity.

The code uses:

```python
X_grid, S_grid = torch.meshgrid(
    grid_x.view(-1),
    fidelity_levels,
    indexing="ij",
)
```

This creates all combinations of:

- each loading value in `grid_x`,
- each fidelity level in `fidelity_levels`.

The final candidate pool therefore contains points such as:

```python
[x, 0.2]
[x, 0.5]
[x, 1.0]
```

for many possible values of $x$.

This is how the workflow makes fidelity an active decision variable.

Instead of fixing $s=1$, the candidate pool explicitly includes multiple possible fidelity levels.

---

### How different fidelities are actively selected

At each BO step, the code first computes the cost of every candidate in the full pool:

```python
candidate_costs = fidelity_cost_from_X(candidate_pool_full)
```

Because each candidate is an augmented input $[x,s]$, the cost is determined by its fidelity level $s$.

Then the code filters out candidates that are too expensive for the remaining budget:

```python
affordable_mask = candidate_costs <= remaining_budget + 1e-12
```

This means the optimiser can only select experiments that still fit within the budget.

Among the affordable candidates, the workflow uses UCB divided by cost:

$$
\mathrm{utility}(x,s)
=
\frac{\mathrm{UCB}(x,s)}{\mathrm{cost}(s)}.
$$

So the selected candidate is not simply the point with the highest predicted activity or uncertainty.

It is the point that gives the best acquisition value **per unit cost**.

That is the mechanism that makes the workflow actively choose between low-, medium-, and high-fidelity evaluations.

Low-fidelity candidates are cheaper, so they can be attractive even if their raw UCB is not the largest.

High-fidelity candidates are more expensive, so they need to be sufficiently valuable to justify their cost.

---

### Why a high-fidelity confirmation rule is included

A pure acquisition-per-cost rule can over-prefer cheap low-fidelity evaluations.

That is because dividing by cost strongly rewards cheap experiments.

However, the final target is still the high-fidelity objective:

$$
f(x,s=1).
$$

So the workflow includes a simple confirmation rule:

```python
should_confirm = step > 0 and step % confirm_every == 0
```

Every few steps, if there is enough remaining budget, the workflow evaluates the current model-recommended loading at high fidelity:

```python
candidate = torch.cat(
    [
        x_rec,
        torch.ones_like(x_rec),
    ],
    dim=-1,
)
```

This forces the candidate to have:

```python
[x_rec, 1.0]
```

So the model occasionally anchors its learning with a true high-fidelity measurement.

This makes the workflow more realistic.

The cheap lower-fidelity evaluations help explore efficiently, but the high-fidelity confirmations keep the model connected to the real target.

---

### What is stored in `history`

As in the high-fidelity-only loop, this workflow stores a history dictionary at each step.

The recorded quantities include:

- cumulative cost,
- number of observations,
- current high-fidelity recommendation,
- true high-fidelity activity at that recommendation,
- posterior mean and uncertainty at the recommendation,
- and the last fidelity level evaluated.

The `last_fidelity` entry is especially useful here because it lets us inspect how often the workflow chose low-, medium-, or high-fidelity evaluations.

This will be plotted later to understand the behaviour of the multi-fidelity policy.

---

### Why the final recommendation is still high fidelity

Even though the loop evaluates a mixture of fidelity levels, the recommendation is always made using:

```python
recommend_high_fidelity_from_model(model, grid_x)
```

This evaluates the model on the target slice:

```python
[x, 1.0]
```

So the final recommendation answers:

> which co-catalyst loading is predicted to be best at high fidelity?

not:

> which low-fidelity measurement looked best?

This distinction is central to multi-fidelity BO.

Low-fidelity evaluations are used to guide learning, but they are not the final optimisation target.

---

### Key takeaway

This cell implements a simple multi-fidelity BO workflow where fidelity is actively selected.

The candidate pool contains multiple fidelity levels, the acquisition value is divided by evaluation cost, and occasional high-fidelity confirmation steps keep the model anchored to the true target.

So the workflow balances three competing priorities:

- cheap exploration at low fidelity,
- more reliable information at high fidelity,
- and total budget control.

In [ ]:
def make_multi_fidelity_candidate_pool(grid_x, fidelity_levels):
    X_grid, S_grid = torch.meshgrid(
        grid_x.view(-1),
        fidelity_levels,
        indexing="ij",
    )

    candidate_X = torch.stack(
        [
            X_grid.reshape(-1),
            S_grid.reshape(-1),
        ],
        dim=-1,
    )

    return candidate_X


def run_multi_fidelity_bo(
    train_X_init,
    train_Y_init,
    budget_total,
    max_steps=40,
    beta=1.5,
    confirm_every=6,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()

    spent = float(fidelity_cost_from_X(train_X).sum())
    history = []

    candidate_pool_full = make_multi_fidelity_candidate_pool(grid_x, fidelity_levels)

    for step in range(max_steps + 1):
        model = fit_augmented_gp(train_X, train_Y)
        x_rec, y_rec_true, mean_rec, std_rec = recommend_high_fidelity_from_model(model, grid_x)

        history.append({
            "step": step,
            "cumulative_cost": spent,
            "n_observations": train_X.shape[0],
            "recommended_x": float(x_rec),
            "recommended_true_high_fidelity": float(y_rec_true),
            "recommendation_mean": float(mean_rec),
            "recommendation_std": float(std_rec),
            "last_fidelity": None if step == 0 else float(train_X[-1, 1]),
            "method": "Multi-fidelity",
        })

        if step == max_steps:
            break

        candidate_costs = fidelity_cost_from_X(candidate_pool_full)
        remaining_budget = budget_total - spent

        affordable_mask = candidate_costs <= remaining_budget + 1e-12

        if not affordable_mask.any():
            break

        candidate_pool = candidate_pool_full[affordable_mask]
        candidate_costs_affordable = candidate_costs[affordable_mask]

        should_confirm = step > 0 and step % confirm_every == 0
        high_fidelity_cost = float(fidelity_cost_from_s(torch.tensor(1.0)))

        if should_confirm and remaining_budget >= high_fidelity_cost:
            x_rec, _, _, _ = recommend_high_fidelity_from_model(model, grid_x)
            candidate = torch.cat(
                [
                    x_rec,
                    torch.ones_like(x_rec),
                ],
                dim=-1,
            )
        else:
            candidate, info = select_ucb_candidate(
                model=model,
                candidate_X=candidate_pool,
                beta=beta,
                utility_denominator=candidate_costs_affordable,
            )

        Y_new = multi_fidelity_objective(candidate)
        next_cost = float(fidelity_cost_from_X(candidate))

        train_X = torch.cat([train_X, candidate], dim=0)
        train_Y = torch.cat([train_Y, Y_new], dim=0)

        spent += next_cost

    final_model = fit_augmented_gp(train_X, train_Y)

    return {
        "history": history,
        "train_X_final": train_X,
        "train_Y_final": train_Y,
        "model_final": final_model,
    }


set_seed(seed)

mf_result = run_multi_fidelity_bo(
    train_X_init=train_X_mf_init,
    train_Y_init=train_Y_mf_init,
    budget_total=budget_total,
    max_steps=40,
    beta=1.5,
    confirm_every=5,
)

print("Multi-fidelity observations:", mf_result["train_X_final"].shape[0])
print("Final cost:", mf_result["history"][-1]["cumulative_cost"])
print("Final recommended x:", mf_result["history"][-1]["recommended_x"])
print("Final true high-fidelity activity:", mf_result["history"][-1]["recommended_true_high_fidelity"])

## 9. Comparing high-fidelity-only and multi-fidelity BO under a budget

This cell compares the two BO workflows using the most important diagnostic in this part of the notebook:

> how good is the current **high-fidelity recommendation** as a function of cumulative evaluation cost?

This is the right comparison because the two workflows do not spend budget in the same way.

The high-fidelity-only workflow evaluates only at $s=1$, so every new observation is accurate but expensive.

The multi-fidelity workflow can evaluate at $s=0.2$, $s=0.5$, or $s=1.0$, so it can collect more information for the same total budget by using cheaper low- and medium-fidelity evaluations.

---

### What is being plotted

The y-axis is:

$$
\text{true high-fidelity activity at the currently recommended } x.
$$

This is important.

The plot is not showing the best low-fidelity value, nor the best observed proxy measurement.

Instead, at each stage, the model recommends a co-catalyst loading based on its predicted high-fidelity performance, and we then evaluate that recommendation against the true high-fidelity objective:

$$
f(x, s=1).
$$

So both methods are judged by the same final target.

---

### What the figure shows

The high-fidelity-only workflow starts at a higher cumulative cost because its initial design already consists of five full high-fidelity evaluations.

Since each high-fidelity evaluation costs $1.0$, the initial high-fidelity-only design already spends:

$$
5.0
$$

out of the total budget of:

$$
8.0.
$$

By contrast, the multi-fidelity workflow starts at a much lower cumulative cost because its initial design mixes low-, medium-, and high-fidelity evaluations.

This allows it to begin the sequential BO phase with more remaining budget.

The figure then shows that the multi-fidelity workflow reaches a near-optimal high-fidelity recommendation very quickly, at a much lower cumulative cost than the high-fidelity-only baseline.

The high-fidelity-only workflow also eventually reaches the optimum region, but it needs more budget because every new evaluation is expensive.

---

### Main conclusion

The key conclusion is not that multi-fidelity BO discovers a fundamentally different optimum.

Both workflows eventually recommend a co-catalyst loading close to the true high-fidelity optimum.

The important difference is **budget efficiency**.

The multi-fidelity workflow reaches a very strong high-fidelity recommendation using many more total observations but at lower cost per observation.

This demonstrates the practical value of multi-fidelity BO:

> low- and medium-fidelity evaluations can help guide the search toward the high-fidelity optimum more cheaply.

---

### Interpreting the summary table

The summary table makes the same point numerically.

It compares:

- the final number of observations,
- the final cumulative cost,
- the final recommended co-catalyst loading,
- and the true high-fidelity activity at that recommendation.

In this run, the multi-fidelity workflow obtains many more observations than the high-fidelity-only workflow while staying within roughly the same budget.

Its final recommended $x$ is extremely close to the true high-fidelity optimum, and its final high-fidelity activity is almost identical to the optimum value.

So the low-fidelity evaluations were not treated as the final answer.

They were used as cheaper information sources to help the model identify the high-fidelity optimum efficiently.

---

### Key takeaway

This comparison shows the main reason multi-fidelity BO is useful.

High-fidelity-only BO gives accurate information, but each evaluation is expensive.

Multi-fidelity BO uses cheaper biased evaluations to learn faster under the same budget, while still judging the final recommendation at high fidelity.

So the main lesson is:

> multi-fidelity BO is valuable when cheap approximate evaluations can guide the search toward a strong high-fidelity recommendation before the budget is exhausted.

In [ ]:
hf_hist = pd.DataFrame(hf_result["history"])
mf_hist = pd.DataFrame(mf_result["history"])

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    hf_hist["cumulative_cost"],
    hf_hist["recommended_true_high_fidelity"],
    "-o",
    lw=2.5,
    markersize=6,
    label="High-fidelity only",
)

ax.plot(
    mf_hist["cumulative_cost"],
    mf_hist["recommended_true_high_fidelity"],
    "-o",
    lw=2.5,
    markersize=6,
    label="Multi-fidelity",
)

ax.axhline(
    float(true_best_y),
    linestyle="--",
    lw=2.0,
    color=TRUE_COLOR,
    label="True high-fidelity optimum",
)

ax.set_title("High-fidelity recommendation vs budget", fontsize=18, fontweight="bold")
ax.set_xlabel("Cumulative evaluation cost", fontsize=22, fontweight="bold")
ax.set_ylabel("True high-fidelity activity \n at recommended x", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

summary_mf_df = pd.DataFrame({
    "method": ["High-fidelity only", "Multi-fidelity"],
    "n_observations": [
        hf_result["train_X_final"].shape[0],
        mf_result["train_X_final"].shape[0],
    ],
    "final_cost": [
        hf_hist["cumulative_cost"].iloc[-1],
        mf_hist["cumulative_cost"].iloc[-1],
    ],
    "final_recommended_x": [
        hf_hist["recommended_x"].iloc[-1],
        mf_hist["recommended_x"].iloc[-1],
    ],
    "final_true_high_fidelity_activity": [
        hf_hist["recommended_true_high_fidelity"].iloc[-1],
        mf_hist["recommended_true_high_fidelity"].iloc[-1],
    ],
})

display(summary_mf_df)

## 10. Inspecting which fidelity levels were selected

This cell inspects the fidelity levels chosen by the multi-fidelity BO workflow.

The goal is to understand how the acquisition-per-cost rule actually behaved during the run.

The dataframe `mf_df` records, for every multi-fidelity evaluation:

- the co-catalyst loading `x`,
- the selected fidelity level,
- the observed activity,
- and the evaluation cost.

The scatter plot then shows the fidelity level selected at each evaluation index.

---

### What the figure shows

Most evaluations are performed at the lowest fidelity level:

$$
s = 0.2.
$$

Only a small number of evaluations are performed at medium fidelity:

$$
s = 0.5,
$$

and several evaluations are performed at high fidelity:

$$
s = 1.0.
$$

The high-fidelity evaluations appear at regular intervals because the loop includes a confirmation rule:

```python
should_confirm = step > 0 and step % confirm_every == 0
```

So the workflow does not rely only on cheap proxy evaluations.

Every few steps, it forces a high-fidelity check of the current model-recommended loading, provided there is enough remaining budget.

This is why the plot contains occasional points at $s=1.0$.

---

### Why low fidelity is selected so often

The main selection rule is acquisition-per-cost:

$$
\mathrm{utility}(x,s)
=
\frac{\mathrm{UCB}(x,s)}{\mathrm{cost}(s)}.
$$

This strongly favours cheap evaluations.

The cost model is:

$$
\mathrm{cost}(s)=0.15+0.85s^2.
$$

So the available fidelity costs are approximately:

- $s=0.2$: cheap,
- $s=0.5$: intermediate,
- $s=1.0$: expensive.

Because $s=0.2$ is much cheaper than $s=1.0$, its UCB does not need to be very large to have a high UCB-per-cost value.

This makes low-fidelity candidates very attractive to the acquisition rule.

---

### Why this is slightly too strict for variables in $[0,1]$

The UCB-per-cost rule can be quite aggressive when the input variables and fidelity levels are scaled to $[0,1]$.

In this notebook, the fidelity cost changes substantially across the interval:

$$
s \in [0,1].
$$

Since the utility divides directly by cost, the low-fidelity candidates receive a strong numerical advantage.

That means the workflow can become overly biased toward cheap evaluations, even when high-fidelity data would be valuable for anchoring the model.

This is why the high-fidelity confirmation rule is useful.

Without it, the acquisition-per-cost policy might select too many low-fidelity points and too few high-fidelity points.

So the figure reveals an important modelling lesson:

> a simple UCB/cost rule is useful, but it can over-prioritise cheap evaluations when the cost contrast is large.

---

### How this could be made less aggressive

There are several ways to soften the cost penalty.

For example, instead of using:

$$
\frac{\mathrm{UCB}}{\mathrm{cost}},
$$

one could use:

$$
\frac{\mathrm{UCB}}{\sqrt{\mathrm{cost}}},
$$

or more generally:

$$
\frac{\mathrm{UCB}}{\mathrm{cost}^{\gamma}}, \quad 0 < \gamma < 1.
$$

This would still prefer cheaper evaluations, but less strongly.

Another option is to require occasional high-fidelity confirmation, as done here.

That keeps the simple UCB-per-cost logic while preventing the model from drifting too far away from the true high-fidelity target.

---

### Key takeaway

This diagnostic shows that the multi-fidelity BO policy actively selects fidelity levels rather than fixing all evaluations at $s=1$.

Most evaluations are cheap low-fidelity probes, while occasional high-fidelity evaluations anchor the model to the true target.

The behaviour also shows a limitation of the simple UCB-per-cost heuristic:

> dividing directly by cost can be too strict, so cheap low-fidelity evaluations may dominate unless the workflow includes a confirmation rule or a softer cost penalty.

In [ ]:
mf_train_X = mf_result["train_X_final"]
mf_train_Y = mf_result["train_Y_final"]

mf_df = pd.DataFrame({
    "x": mf_train_X[:, 0].detach().cpu().numpy(),
    "fidelity": mf_train_X[:, 1].detach().cpu().numpy(),
    "observed_activity": mf_train_Y.view(-1).detach().cpu().numpy(),
    "cost": fidelity_cost_from_X(mf_train_X).detach().cpu().numpy(),
})

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    mf_df.index,
    mf_df["fidelity"],
    s=90,
    edgecolor="black",
    lw=1.1,
)

ax.set_title("Fidelity levels selected by multi-fidelity BO", fontsize=18, fontweight="bold")
ax.set_xlabel("Evaluation index", fontsize=22, fontweight="bold")
ax.set_ylabel("Fidelity level", fontsize=22, fontweight="bold")
style_ax(ax)
plt.show()

## 11. Final high-fidelity recommendations and posterior comparison

This cell compares the final models learned by the two workflows:

- **high-fidelity-only BO**,
- and **multi-fidelity BO**.

The goal is to separate two related but different questions:

1. What final co-catalyst loading does each method recommend?
2. How well does each model reconstruct the high-fidelity target curve?

That distinction matters because a method can make a good final recommendation even if its posterior mean is not perfect everywhere.

---

### Final recommendation quality

The left panel shows the true high-fidelity target together with the final recommendation from each workflow.

Both methods recommend a co-catalyst loading very close to the true high-fidelity optimum.

The table confirms this numerically:

- the high-fidelity-only recommendation is close to the optimum,
- the multi-fidelity recommendation is also close to the optimum,
- and both achieve almost the same true high-fidelity activity.

So the main conclusion from the left panel is:

> both workflows successfully identify the high-fidelity optimum region.

This is important because the multi-fidelity workflow used many low- and medium-fidelity evaluations, but its final recommendation is still judged against the true high-fidelity target.

It is not being rewarded for finding a good low-fidelity proxy value.

---

### Posterior means on the high-fidelity slice

The right panel compares the learned posterior means on the slice:

$$
s = 1.
$$

This means both models are being evaluated as predictions of the high-fidelity target.

The high-fidelity-only posterior is trained entirely on $s=1$ observations, so it is strongly anchored to the true target.

The multi-fidelity posterior is trained on a mixture of:

- low-fidelity observations,
- medium-fidelity observations,
- and occasional high-fidelity confirmations.

This means the multi-fidelity model has to learn the relationship between low-fidelity proxy data and the true high-fidelity target.

In this particular run, the multi-fidelity posterior still tracks the high-fidelity target well near the optimum, which is why it gives a good final recommendation.

However, some low-fidelity observations are visibly displaced from the high-fidelity curve.

That is expected.

They are not failed high-fidelity measurements; they are biased proxy evaluations.

---

### Why the plotted recommendations are slightly offset

In the left panel, the two recommendation markers are shifted slightly in the vertical direction.

This is only for visual clarity, because the two recommendations are extremely close to each other and would otherwise overlap.

The true high-fidelity values in the summary table are computed without this visual offset.

So the offset should not be interpreted as a real change in activity.

---

### What this example demonstrates

This example shows the ideal use case for multi-fidelity BO.

The lower-fidelity evaluations are biased, but still informative enough to guide the model toward the correct high-fidelity region.

As a result, the multi-fidelity workflow reaches a near-optimal high-fidelity recommendation while collecting many more evaluations under the same budget.

So the useful lesson is:

> multi-fidelity BO can be efficient when cheap proxy evaluations are correlated with the high-fidelity target.

---

### Important limitation

This is still a **heuristic teaching example**.

The multi-fidelity workflow here uses a simple augmented-input GP and a UCB-per-cost acquisition rule.

It does not use a fully principled multi-fidelity acquisition function, and it does not explicitly model the fidelity relationship using a specialised multi-fidelity kernel.

Also, the low-fidelity functions in this synthetic example are still reasonably informative.

In real experimental work, low-fidelity data may be much worse.

For example, a proxy assay may:

- have a different optimum,
- exaggerate the wrong region,
- miss an important failure mode,
- be poorly correlated with the full experiment,
- or become misleading outside a narrow range of conditions.

In those cases, multi-fidelity BO may not perform this cleanly.

Low-fidelity evaluations are only valuable if they provide useful information about the high-fidelity target.

---

### Key takeaway

The final conclusion is not that multi-fidelity BO is automatically better than high-fidelity-only BO.

The more careful conclusion is:

> when low-fidelity evaluations are cheap and sufficiently informative, they can help BO identify a strong high-fidelity recommendation more efficiently.

In this synthetic example, that condition is satisfied.

But in real applications, the quality of the low-fidelity proxy is the central question.

In [ ]:
hf_model = hf_result["model_final"]
mf_model = mf_result["model_final"]

target_grid_aug = torch.cat(
    [
        grid_x,
        torch.ones_like(grid_x),
    ],
    dim=-1,
)

with torch.no_grad():
    hf_posterior = hf_model.posterior(target_grid_aug)
    hf_mean = hf_posterior.mean.squeeze(-1)
    hf_std = hf_posterior.variance.sqrt().squeeze(-1)

    mf_posterior = mf_model.posterior(target_grid_aug)
    mf_mean = mf_posterior.mean.squeeze(-1)
    mf_std = mf_posterior.variance.sqrt().squeeze(-1)

hf_x_rec = torch.tensor([[hf_hist["final_recommended_x"].iloc[-1] if "final_recommended_x" in hf_hist.columns else hf_hist["recommended_x"].iloc[-1]]], dtype=torch.double)
mf_x_rec = torch.tensor([[mf_hist["final_recommended_x"].iloc[-1] if "final_recommended_x" in mf_hist.columns else mf_hist["recommended_x"].iloc[-1]]], dtype=torch.double)

hf_y_rec = high_fidelity_objective(hf_x_rec)
mf_y_rec = high_fidelity_objective(mf_x_rec)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    grid_x.view(-1),
    grid_y_high.view(-1),
    lw=2.8,
    color=TRUE_COLOR,
    label="True high-fidelity target",
)

axes[0].scatter(
    hf_x_rec.view(-1),
    hf_y_rec.view(-1)-0.01,
    s=150,
    edgecolor="black",
    lw=1.2,
    label="HF-only recommendation",
)

axes[0].scatter(
    mf_x_rec.view(-1),
    mf_y_rec.view(-1)+0.01,
    s=150,
    marker="s",
    edgecolor="black",
    lw=1.2,
    label="MF recommendation",
)

axes[0].axvline(float(true_best_x), linestyle="--", lw=2.0, color=TRUE_COLOR, label="True optimum")

axes[0].set_title("Final high-fidelity recommendations", fontsize=18, fontweight="bold")
axes[0].set_xlabel("Co-catalyst loading", fontsize=22, fontweight="bold")
axes[0].set_ylabel("True high-fidelity activity", fontsize=22, fontweight="bold")
axes[0].legend(prop={"size": 10, "weight": "bold"})
style_ax(axes[0])

axes[1].plot(
    grid_x.view(-1),
    grid_y_high.view(-1),
    lw=2.8,
    color=TRUE_COLOR,
    label="True high-fidelity target",
)

axes[1].plot(
    grid_x.view(-1),
    hf_mean.view(-1),
    lw=2.4,
    label="HF-only posterior mean at s = 1",
)

axes[1].plot(
    grid_x.view(-1),
    mf_mean.view(-1),
    lw=2.4,
    label="MF posterior mean at s = 1",
)

hf_obs_X = hf_result["train_X_final"]
hf_obs_Y = hf_result["train_Y_final"]

mf_obs_X = mf_result["train_X_final"]
mf_obs_Y = mf_result["train_Y_final"]
mf_high_mask = torch.isclose(mf_obs_X[:, 1], torch.tensor(1.0, dtype=torch.double))
mf_low_mask = ~mf_high_mask

axes[1].scatter(
    hf_obs_X[:, 0],
    hf_obs_Y.view(-1)-0.01,
    s=50,
    edgecolor="black",
    lw=1.0,
    label="HF-only s = 1 observations",
)

axes[1].scatter(
    mf_obs_X[mf_high_mask, 0],
    mf_obs_Y[mf_high_mask].view(-1)+0.01,
    s=50,
    marker="s",
    edgecolor="black",
    lw=1.0,
    label="MF s = 1 observations",
)

axes[1].scatter(
    mf_obs_X[mf_low_mask, 0],
    mf_obs_Y[mf_low_mask].view(-1),
    s=45,
    marker="s",
    alpha=0.5,
    edgecolor="black",
    lw=0.8,
    label="MF low-fidelity observations",
)

axes[1].axvline(float(true_best_x), linestyle="--", lw=2.0, color=TRUE_COLOR, label="True optimum")

axes[1].set_title("Posterior means on the high-fidelity slice", fontsize=18, fontweight="bold")
axes[1].set_xlabel("Co-catalyst loading", fontsize=22, fontweight="bold")
axes[1].set_ylabel("Activity", fontsize=22, fontweight="bold")
axes[1].legend(prop={"size": 10, "weight": "bold"})
style_ax(axes[1])

plt.tight_layout()
plt.show()

recommendation_comparison_df = pd.DataFrame({
    "method": ["High-fidelity only", "Multi-fidelity"],
    "recommended_x": [float(hf_x_rec), float(mf_x_rec)],
    "true_high_fidelity_activity_at_recommendation": [float(hf_y_rec), float(mf_y_rec)],
    "distance_from_true_optimum_x": [
        abs(float(hf_x_rec) - float(true_best_x)),
        abs(float(mf_x_rec) - float(true_best_x)),
    ],
})

display(recommendation_comparison_df)

## 12. Defining a contextual photocatalysis objective

This cell begins the **contextual BO** part of the notebook.

So far, the tutorial has focused on **multi-fidelity BO**, where the optimiser chooses both:

```python
[x, s]
```

where:

- $x$ is the co-catalyst loading,
- $s$ is the fidelity level.

In that setting, $s$ controls the **accuracy and cost** of the evaluation.

Now we introduce a different kind of extra variable:

```python
[x, c]
```

where:

- $x$ is the co-catalyst loading,
- $c$ is the **light-intensity context**.

The important difference is that $c$ is not a fidelity level.

It does not mean “cheap” or “expensive”.

Instead, it represents the external condition under which the photocatalyst is tested.

---

### What contextual BO means

In ordinary BO, we usually model an objective such as:

$$
f(x),
$$

and ask:

> which $x$ gives the best value?

In contextual BO, the objective depends on both a design variable and a context:

$$
f(x,c).
$$

The question becomes:

> given the current context $c$, which design $x$ should we choose?

So contextual BO is not necessarily trying to find one universally best co-catalyst loading.

Instead, it tries to learn how the best loading changes as the context changes.

In this example, the context is light intensity.

A low-light condition may favour one co-catalyst loading, while a high-light condition may favour another.

That is why contextual BO is useful: it can make recommendations that are conditional on the operating environment.

---

### How the optimum changes with context

The line

```python
x_star = 0.25 + 0.55 * c
```

defines the main idea of the synthetic example.

It means that the location of the main activity peak shifts as the context changes.

When $c$ is small, the best co-catalyst loading is closer to the lower-loading region.

When $c$ is large, the best co-catalyst loading moves toward a higher-loading region.

So the true optimum is no longer a single fixed value of $x$.

Instead, the optimal loading becomes context-dependent:

$$
x^*(c).
$$

This gives the notebook a clean way to demonstrate why contextual BO is different from standard BO.

---

### What the objective function contains

The function `contextual_photocatalysis_objective(X)` contains three main parts.

First, the main context-dependent peak:

```python
main_peak = torch.exp(-0.5 * ((x - x_star) / 0.13) ** 2)
```

This is the dominant activity region, whose location shifts with $c$.

Second, a smaller secondary peak:

```python
secondary_peak = 0.25 * torch.exp(
    -0.5 * ((x - (0.75 - 0.25 * c)) / 0.18) ** 2
)
```

This makes the landscape slightly richer, so the model is not just learning one simple Gaussian bump.

Third, the context-dependent scaling:

```python
context_scale = 0.75 + 0.35 * c
```

This means that the activity level itself can also change with context, not only the best location.

Together, these terms create a synthetic photocatalysis objective where both the shape and optimum depend on light intensity.

---

### Difference from multi-objective BO

This is not multi-objective BO.

In multi-objective BO, we optimise several outputs at the same time, for example:

$$
\left(f_1(x), f_2(x)\right).
$$

The goal is usually to learn a Pareto frontier of trade-offs between objectives.

For example, in the previous tutorial, the optimiser had to balance Coulombic efficiency and charging capacity.

In contextual BO, there is still only one output:

$$
f(x,c).
$$

The complication is not that there are multiple objectives.

The complication is that the best design depends on the context.

So:

- **multi-objective BO** asks: how do we trade off several objectives?
- **contextual BO** asks: given this context, which design is best?

---

### Difference from multi-fidelity BO

This is also different from multi-fidelity BO.

In multi-fidelity BO, the extra variable $s$ controls evaluation quality and cost:

$$
f(x,s).
$$

The target is still the high-fidelity slice:

$$
f(x,s=1).
$$

Low-fidelity evaluations are useful only because they provide cheaper information about that final target.

In contextual BO, the extra variable $c$ is not a cheaper approximation.

It is the condition under which the design is judged.

There is usually no single “highest context” that acts as the universal target.

Instead, different contexts may genuinely require different optimal designs.

So:

- **multi-fidelity BO** asks: should I evaluate cheaply or accurately?
- **contextual BO** asks: under this condition, which design should I choose?

---

### Why we define example contexts

The line

```python
context_values_to_plot = torch.tensor([0.1, 0.5, 0.9], dtype=torch.double)
```

selects three representative light-intensity contexts:

- low context,
- medium context,
- high context.

These will be used in the next cell to visualise how the activity curve changes with context.

This visualisation is important because it will show that the optimum shifts as $c$ changes.

---

### Key takeaway

This cell defines a contextual photocatalysis objective where the best co-catalyst loading depends on light intensity.

The main conceptual shift is:

> contextual BO does not search for one universal best design; it learns how the best design changes with the context.

This is distinct from both multi-objective BO and multi-fidelity BO:

- multi-objective BO handles multiple outputs,
- multi-fidelity BO handles different evaluation qualities and costs,
- contextual BO handles changing external conditions.

In [ ]:
def contextual_photocatalysis_objective(X):
    if X.ndim == 1:
        X = X.unsqueeze(0)

    x = X[..., 0]
    c = X[..., 1]

    x_star = 0.25 + 0.55 * c

    main_peak = torch.exp(-0.5 * ((x - x_star) / 0.13) ** 2)

    secondary_peak = 0.25 * torch.exp(
        -0.5 * ((x - (0.75 - 0.25 * c)) / 0.18) ** 2
    )

    context_scale = 0.75 + 0.35 * c

    y = (
        0.25
        + context_scale * main_peak
        + secondary_peak
        + 0.04 * torch.sin(12.0 * x + 4.0 * c)
    )

    return y.unsqueeze(-1)


context_values_to_plot = torch.tensor([0.1, 0.5, 0.9], dtype=torch.double)

print("Example contexts:", context_values_to_plot.tolist())

## 13. Visualising how the optimum changes with context

This cell visualises the contextual photocatalysis objective at three different light-intensity contexts.

The purpose is to show why this is a **contextual BO** problem rather than an ordinary single-objective BO problem.

For each context value $c$, the code constructs inputs of the form:

```python
[x, c]
```

where:

- $x$ is the co-catalyst loading,
- $c$ is the light-intensity context.

The function `contextual_photocatalysis_objective(...)` is then evaluated across the full loading grid for each fixed context.

So each curve shows:

> how photocatalytic activity changes with co-catalyst loading under one specific light-intensity condition.

---

### What the figure shows

The figure shows a drastic change in the activity landscape across different contexts.

When the context is low, for example $c = 0.1$, the best co-catalyst loading is around:

$$
x \approx 0.30.
$$

When the context is intermediate, $c = 0.5$, the optimum shifts to around:

$$
x \approx 0.53.
$$

When the context is high, $c = 0.9$, the optimum shifts further to around:

$$
x \approx 0.74.
$$

So the best loading is not fixed.

It moves substantially as the light-intensity context changes.

This is the main point of the figure.

---

### Why this matters

If we ignored context, we might try to find one globally best co-catalyst loading.

But this figure shows that such a single universal recommendation would be incomplete.

A loading that is good under one context may not be optimal under another.

For example, a lower co-catalyst loading is best under low light intensity, while a higher loading becomes optimal under high light intensity.

So the optimisation question should not be:

> what is the best co-catalyst loading overall?

It should be:

> given the current light-intensity context, what co-catalyst loading should we choose?

That is exactly the contextual BO setting.

---

### Context changes both optimum location and activity scale

The context does not only shift the location of the optimum.

It also changes the activity values themselves.

The curves differ in both:

- the position of their main peak,
- and the height and shape of the activity landscape.

This means the context affects the whole response surface, not just the best loading.

That is why the model needs to learn a function of both variables:

$$
f(x,c),
$$

rather than only:

$$
f(x).
$$

---

### Key takeaway

This figure demonstrates the core motivation for contextual BO.

The optimal co-catalyst loading changes dramatically with light-intensity context, so the model should make **context-dependent recommendations**.

In other words:

> contextual BO learns not just which design is good, but which design is good under a particular external condition.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for c in context_values_to_plot:
    X_context = torch.cat(
        [
            grid_x,
            torch.full_like(grid_x, float(c)),
        ],
        dim=-1,
    )

    y_context = contextual_photocatalysis_objective(X_context)

    best_idx = torch.argmax(y_context)
    best_x = grid_x[best_idx]

    ax.plot(
        grid_x.view(-1),
        y_context.view(-1),
        lw=2.5,
        label=f"context c = {float(c):.1f}, best x = {float(best_x):.2f}",
    )

ax.set_title("Context changes the optimal co-catalyst loading", fontsize=18, fontweight="bold")
ax.set_ylabel("Recommended co-catalyst loading", fontsize=22, fontweight="bold")
ax.set_ylabel("Contextual activity", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 14. Initial contextual dataset

This cell creates the initial dataset for the contextual BO workflow.

In the contextual part of the notebook, each input has the form:

```python
[x, c]
```

where:

- $x$ is the co-catalyst loading,
- $c$ is the light-intensity context.

So unlike ordinary BO, the model is not only learning how activity changes with co-catalyst loading.

It is learning the full contextual function:

$$
f(x,c).
$$

---

### Sampling initial contextual observations

The code first defines a two-dimensional input space:

$$
(x,c) \in [0,1]^2.
$$

This is done using:

```python
context_bounds = torch.tensor(
    [[0.0, 0.0], [1.0, 1.0]],
    dtype=torch.double,
)
```

The first dimension corresponds to co-catalyst loading.

The second dimension corresponds to light-intensity context.

Then a Sobol design is used to generate the initial contextual observations:

```python
train_X_context_init = draw_sobol_samples(
    bounds=context_bounds,
    n=1,
    q=n_context_init,
).squeeze(0)
```

This gives an initial set of points spread across both loading and context.

That is useful because the model needs to learn not only where good loadings are, but also how the response changes as the context changes.

---

### Evaluating the contextual objective

The initial observations are evaluated using:

```python
train_Y_context_init = contextual_photocatalysis_objective(train_X_context_init)
```

Each row of `train_X_context_init` contains a different pair:

```python
[x, c]
```

and the output is the corresponding contextual activity:

$$
y = f(x,c).
$$

So the initial dataset gives the GP a first rough picture of the two-dimensional contextual response surface.

---

### Why this initial dataset matters

In contextual BO, the model needs examples across different contexts.

If all initial observations were collected at only one context, the GP would have little information about how the optimal loading changes as context changes.

By sampling across the full $[0,1]^2$ space, the initial dataset gives the model information about:

- different co-catalyst loadings,
- different light-intensity contexts,
- and how activity varies jointly with both.

This helps the later BO loop make context-dependent recommendations.

---

### What the dataframe shows

The dataframe displays the initial contextual observations in a readable form.

It contains:

- `x`: the sampled co-catalyst loading,
- `context_light_intensity`: the sampled light-intensity context,
- `activity`: the observed contextual activity.

This table is a sanity check that the initial design covers both input dimensions and that each contextual observation has been evaluated correctly.

---

### Key takeaway

This cell creates the starting dataset for contextual BO.

The important point is that contextual BO learns from observations of the form:

$$
(x,c,y),
$$

not just:

$$
(x,y).
$$

So the model can learn how the best co-catalyst loading changes with light-intensity context.

In [ ]:
set_seed(seed + 100)

context_bounds = torch.tensor(
    [[0.0, 0.0], [1.0, 1.0]],
    dtype=torch.double,
)

n_context_init = 10

train_X_context_init = draw_sobol_samples(
    bounds=context_bounds,
    n=1,
    q=n_context_init,
).squeeze(0)

train_Y_context_init = contextual_photocatalysis_objective(train_X_context_init)

context_init_df = pd.DataFrame({
    "x": train_X_context_init[:, 0].detach().cpu().numpy(),
    "context_light_intensity": train_X_context_init[:, 1].detach().cpu().numpy(),
    "activity": train_Y_context_init.view(-1).detach().cpu().numpy(),
})

display(context_init_df)

## 15. Contextual BO with fixed observed contexts

This cell defines and runs the **contextual BO loop**.

The key difference from standard BO is that the optimiser does not search for one universally best co-catalyst loading.

Instead, at each BO step, a light-intensity context is given, and the optimiser chooses the best co-catalyst loading **conditional on that context**.

So the decision is not simply:

> which $x$ should we evaluate next?

but rather:

> given the current context $c$, which co-catalyst loading $x$ should we evaluate next?

---

### How the contextual BO loop works

The training inputs have the form:

```python
[x, c]
```

where:

- $x$ is the co-catalyst loading,
- $c$ is the light-intensity context.

The GP therefore models the contextual objective:

$$
f(x,c).
$$

At each BO step, the loop receives one context value from `context_sequence`.

For that fixed context, the code constructs a candidate grid:

```python
context_column = torch.full_like(grid_x, float(context_value))
candidate_X = torch.cat([grid_x, context_column], dim=-1)
```

This creates many candidate points of the form:

```python
[x, c_t]
```

where $c_t$ is the current observed context.

So during that step, BO only searches over $x$ while holding the context fixed.

---

### Candidate selection by UCB

The next candidate is selected using:

```python
candidate, info = select_ucb_candidate(
    model=model,
    candidate_X=candidate_X,
    beta=beta,
)
```

This applies the UCB rule over the fixed-context candidate grid:

$$
\mathrm{UCB}(x,c_t)=\mu(x,c_t)+\beta\sigma(x,c_t).
$$

So the selected loading may be one that is predicted to have high activity, one that is uncertain, or a balance of both.

This is the BO part of the contextual loop.

The optimiser is still balancing exploration and exploitation, but only among candidates that share the current context.

---

### Recommendation before update

After selecting and evaluating the UCB candidate, the code also computes the model's current best recommendation for the same context.

This is done by taking the posterior mean over the fixed-context grid and selecting:

$$
x_{\mathrm{rec}}(c_t)=\arg\max_x \mu(x,c_t).
$$

This recommendation is different from the UCB candidate.

The UCB candidate is the point chosen for the next experiment, because it includes uncertainty.

The recommendation is the point the model currently believes is best, based only on posterior mean.

This distinction is important in BO:

- **candidate** = what we choose to evaluate next,
- **recommendation** = what the model currently believes is best.

---

### Why the context sequence controls the number of steps

The number of BO steps is controlled by the number of supplied contexts.

Here:

```python
n_context_steps = 12
```

and the code checks:

```python
assert len(context_sequence) == n_context_steps
```

This is a useful way to structure contextual BO because each step corresponds to one experimental situation.

The context is not freely optimised by the model.

It is provided to the model, and the model chooses the best loading for that context.

So the loop naturally runs once per context value.

---

### Why contextual BO is harder than standard BO

In standard BO, the model usually learns a function of one design input:

$$
f(x).
$$

The optimiser can focus on finding one best region in $x$.

In contextual BO, the model must learn a response surface over both design and context:

$$
f(x,c).
$$

This is harder for several reasons.

First, the best $x$ may change with $c$.

So there may not be one globally best co-catalyst loading.

Second, the model has to generalise across contexts.

Data collected at one context may help at nearby contexts, but it may not fully determine the optimum at a very different context.

Third, the optimiser must make a conditional decision at every step.

It cannot simply ask where the objective is globally high. It must ask where the objective is high for the current context.

So contextual BO is more demanding because it has to learn not only where good designs are, but how the definition of a good design changes with the environment.

---

### What is stored in `history`

For each contextual BO step, the history records:

- the context value,
- the BO-selected candidate loading,
- the observed activity at that candidate,
- the model recommendation before updating,
- the true activity at that recommendation,
- and posterior uncertainty at the recommended point.

This makes it possible to inspect both the active BO decisions and the model's context-dependent recommendations.

---

### Key takeaway

This cell implements contextual BO by fixing the context at each step and optimising the acquisition function only over co-catalyst loading.

The main idea is:

> contextual BO learns a model of $f(x,c)$, but each decision asks for the best $x$ under the current context $c$.

Compared with standard BO, the challenge is that the optimum is no longer a single fixed point. It can move as the context changes.

In [ ]:
def run_contextual_bo(
    train_X_init,
    train_Y_init,
    context_sequence,
    beta=1.5,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()

    history = []

    for step, context_value in enumerate(context_sequence, start=1):
        model = fit_augmented_gp(train_X, train_Y)

        context_column = torch.full_like(grid_x, float(context_value))
        candidate_X = torch.cat(
            [
                grid_x,
                context_column,
            ],
            dim=-1,
        )

        candidate, info = select_ucb_candidate(
            model=model,
            candidate_X=candidate_X,
            beta=beta,
        )

        Y_new = contextual_photocatalysis_objective(candidate)

        with torch.no_grad():
            posterior = model.posterior(candidate_X)
            mean = posterior.mean.squeeze(-1)
            std = posterior.variance.sqrt().squeeze(-1)

        best_idx = torch.argmax(mean)
        x_rec = grid_x[best_idx:best_idx + 1]
        y_rec_true = contextual_photocatalysis_objective(
            torch.cat(
                [
                    x_rec,
                    torch.tensor([[float(context_value)]], dtype=torch.double),
                ],
                dim=-1,
            )
        )

        history.append({
            "step": step,
            "context": float(context_value),
            "candidate_x": float(candidate[0, 0]),
            "observed_y": float(Y_new),
            "recommended_x_before_update": float(x_rec),
            "recommended_true_y_before_update": float(y_rec_true),
            "posterior_mean_at_recommendation": float(mean[best_idx]),
            "posterior_std_at_recommendation": float(std[best_idx]),
        })

        train_X = torch.cat([train_X, candidate], dim=0)
        train_Y = torch.cat([train_Y, Y_new], dim=0)

    final_model = fit_augmented_gp(train_X, train_Y)

    return {
        "history": history,
        "train_X_final": train_X,
        "train_Y_final": train_Y,
        "model_final": final_model,
    }


n_context_steps = 12

context_sequence = torch.tensor(
    [0.15, 0.85, 0.35, 0.65, 0.25, 0.75, 0.45, 0.95, 0.10, 0.55, 0.70, 0.30],
    dtype=torch.double,
)

assert len(context_sequence) == n_context_steps

set_seed(seed + 200)

context_result = run_contextual_bo(
    train_X_init=train_X_context_init,
    train_Y_init=train_Y_context_init,
    context_sequence=context_sequence,
    beta=1.5,
)

context_history_df = pd.DataFrame(context_result["history"])
display(context_history_df)

## 16. Final contextual BO recommendations against true contextual curves

This cell evaluates the final contextual BO model across several fixed light-intensity contexts.

The goal is to check whether the model has learned not only a generally good co-catalyst loading, but a **context-dependent recommendation rule**:

$$
c \mapsto x_{\mathrm{rec}}(c).
$$

For each test context, the figure compares:

- the true contextual activity curve,
- the GP posterior mean,
- the true best co-catalyst loading,
- the model-recommended loading,
- and the true activity achieved at that recommendation.

This makes the result much easier to interpret than a table alone.

---

### What each panel shows

Each subplot fixes one context value $c$ and plots activity as a function of co-catalyst loading $x$.

So each panel is a one-dimensional slice through the contextual objective:

$$
f(x,c).
$$

The cyan curve shows the true contextual activity curve.

The model posterior mean shows what the final GP has learned from the initial contextual data and the BO-selected observations.

The dashed vertical line marks the true best loading for that context, while the dotted vertical line marks the model-recommended loading.

If the two vertical lines are close, then contextual BO has recommended a loading close to the true optimum for that context.

---

### Why the middle contexts align better

The model performs best for the middle contexts, especially around:

$$
c = 0.4,\quad c = 0.6,\quad c = 0.8.
$$

In these panels, the posterior mean follows the true contextual curve closely near the peak, and the model-recommended loading is almost aligned with the true best loading.

This makes sense because the context settings used during BO cover the interior of the context space relatively well.

The contextual BO sequence was:

```python
[0.15, 0.85, 0.35, 0.65, 0.25, 0.75, 0.45, 0.95, 0.10, 0.55, 0.70, 0.30]
```

These values provide several observations across the middle of the context domain.

As a result, the GP can interpolate reasonably well between observed contexts.

---

### Why the edge contexts deviate more

The edge contexts,

$$
c = 0.0 \quad \text{and} \quad c = 1.0,
$$

are not directly covered by the BO context sequence.

The closest contexts are:

- near $c=0$: $0.10$ and $0.15$,
- near $c=1$: $0.95$.

So at the boundaries, the model has to extrapolate more than it does in the middle of the context range.

That is why the posterior mean can deviate more from the true contextual curve at $c=0.0$ and $c=1.0$.

The uncertainty band is also wider in some edge regions, reflecting weaker support from nearby observations.

This is an important practical point:

> contextual BO is strongest where the context space is well covered, and weaker near poorly sampled boundaries.

---

### How to interpret the summary table

The summary table reports, for each test context:

- `recommended_x`: the loading recommended by the final model,
- `true_activity_at_recommendation`: the true activity achieved at that loading,
- `true_best_x_on_grid`: the actual best loading on the dense grid,
- `true_best_activity_on_grid`: the best achievable activity for that context,
- `absolute_x_error`: how far the recommended loading is from the true best loading,
- `activity_regret`: how much activity is lost relative to the true best loading.

The most important diagnostic is `activity_regret`:

$$
\text{regret}(c)
=
f(x^*(c),c)-f(x_{\mathrm{rec}}(c),c),
$$

where $x^*(c)$ is the true best loading and $x_{\mathrm{rec}}(c)$ is the model recommendation.

A small activity regret means the model recommendation is close to optimal for that context.

---

### Key takeaway

This diagnostic shows that contextual BO has learned the main context-dependent trend: as the light-intensity context changes, the recommended co-catalyst loading also changes.

The model aligns well with the true contextual curves in the better-covered interior of the context space, while the boundary contexts show larger deviations because they are less directly sampled.

So the main lesson is:

> contextual BO can give strong context-dependent recommendations, but its accuracy depends on how well the context space is covered by the initial design and subsequent observed contexts.

In [ ]:
final_context_model = context_result["model_final"]

test_contexts = torch.tensor([0.0, 0.2, 0.4, 0.6, 0.8, 1.0], dtype=torch.double)

rows = []

n_cols = 3
n_rows = int(np.ceil(len(test_contexts) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 9), sharey=True)
axes = axes.flatten()

for ax, c in zip(axes, test_contexts):
    context_grid_X = torch.cat(
        [
            grid_x,
            torch.full_like(grid_x, float(c)),
        ],
        dim=-1,
    )

    true_y_grid = contextual_photocatalysis_objective(context_grid_X)
    true_best_idx = torch.argmax(true_y_grid)
    true_best_x_context = grid_x[true_best_idx]
    true_best_y_context = true_y_grid[true_best_idx]

    with torch.no_grad():
        posterior = final_context_model.posterior(context_grid_X)
        posterior_mean = posterior.mean.squeeze(-1)
        posterior_std = posterior.variance.sqrt().squeeze(-1)

    model_best_idx = torch.argmax(posterior_mean)
    x_rec = grid_x[model_best_idx]

    y_rec_true = contextual_photocatalysis_objective(
        torch.tensor([[float(x_rec), float(c)]], dtype=torch.double)
    )

    ax.plot(
        grid_x.view(-1),
        true_y_grid.view(-1),
        lw=2.6,
        color=TRUE_COLOR,
        label="True contextual curve",
    )

    ax.plot(
        grid_x.view(-1),
        posterior_mean.view(-1),
        lw=2.2,
        label="Model posterior mean",
    )

    ax.fill_between(
        grid_x.view(-1).detach().cpu().numpy(),
        (posterior_mean - 2 * posterior_std).detach().cpu().numpy(),
        (posterior_mean + 2 * posterior_std).detach().cpu().numpy(),
        alpha=0.15,
        label="±2 posterior std" if ax == axes[0] else None,
    )

    ax.axvline(
        float(true_best_x_context),
        linestyle="--",
        lw=2.0,
        color=TRUE_COLOR,
        label="True best x" if ax == axes[0] else None,
    )

    ax.axvline(
        float(x_rec),
        linestyle=":",
        lw=2.2,
        label="Model recommended x" if ax == axes[0] else None,
    )

    ax.scatter(
        float(x_rec),
        float(y_rec_true),
        s=110,
        edgecolor="black",
        lw=1.1,
        zorder=4,
        label="True activity at recommendation" if ax == axes[0] else None,
    )

    ax.set_title(f"context c = {float(c):.1f}", fontsize=16, fontweight="bold")
    ax.set_xlabel("Co-catalyst loading", fontsize=16, fontweight="bold")
    style_ax(ax)

    rows.append({
        "context": float(c),
        "recommended_x": float(x_rec),
        "true_activity_at_recommendation": float(y_rec_true),
        "true_best_x_on_grid": float(true_best_x_context),
        "true_best_activity_on_grid": float(true_best_y_context),
        "absolute_x_error": abs(float(x_rec) - float(true_best_x_context)),
        "activity_regret": float(true_best_y_context - y_rec_true),
    })


for row_start in range(0, len(axes), n_cols):
    axes[row_start].set_ylabel("Contextual activity", fontsize=18, fontweight="bold")
axes[0].legend(prop={"size": 8, "weight": "bold"})

plt.suptitle(
    "Final contextual BO recommendations against true contextual curves",
    fontsize=22,
    fontweight="bold",
)

plt.tight_layout()
plt.show()

context_summary_df = pd.DataFrame(rows)
display(context_summary_df)

## 🧭 Closing Remarks

In this tutorial, we moved from **multi-objective Bayesian Optimisation** to two other ways in which realistic BO workflows become more structured:

- **multi-fidelity BO**, where evaluations differ in accuracy and cost,
- and **contextual BO**, where the best design depends on an external condition.

The central idea was that practical BO is not always just about choosing the next design variable $x$.

Sometimes the optimiser must also decide how accurately to evaluate a candidate, how much budget that evaluation should consume, and whether the best design changes when the experimental context changes.

So the usual BO question:

> **which candidate should we evaluate next?**

became two related but different questions.

In the multi-fidelity part:

> **which design should we evaluate, and at what fidelity?**

In the contextual part:

> **given the current context, which design should we evaluate?**

---

### Multi-fidelity BO

The first half of the notebook used a synthetic photocatalyst optimisation problem where the design variable was **co-catalyst loading**.

The target was the high-fidelity objective:

$$
f(x,s=1).
$$

Lower-fidelity evaluations were introduced as cheaper but biased approximations of that target, so the model learned:

$$
f(x,s),
$$

where $s$ controlled the **quality and cost** of evaluation.

The workflow logic was:

- Chooses both the design x and the fidelity level s.
- Low fidelity is cheaper but biased.
- The final target is still high-fidelity performance at s = 1.

This distinction is important.

A low-fidelity value can be useful for learning, but it is not the final performance criterion.

The final recommendation must still be judged on the high-fidelity slice:

$$
s=1.
$$

The main result was that multi-fidelity BO reached a near-optimal high-fidelity recommendation while collecting many more observations under roughly the same budget.

So the lesson was not that multi-fidelity BO always finds a better optimum.

The better lesson was:

> multi-fidelity BO can be valuable because it uses cheaper approximate evaluations to reach a strong high-fidelity recommendation more efficiently.

At the same time, this was only a heuristic teaching example.

The simple UCB-per-cost rule can over-select cheap evaluations, and real low-fidelity measurements may be much less informative than the synthetic ones used here.

A cheap proxy is only useful if it is sufficiently correlated with the high-fidelity target.

---

### Contextual BO

The second half of the notebook introduced a different augmented input:

$$
(x,c),
$$

where $c$ was the **light-intensity context**.

This gave a contextual objective:

$$
f(x,c).
$$

At first glance, this looks similar to multi-fidelity BO, but the meaning is different.

In contextual BO, the context is not a cheaper or more expensive version of the same experiment.

It is the condition under which the design is judged.

The workflow logic was:

- Observes or receives the context c.
- Chooses the design x conditional on that context.
- The best design can change as the context changes.


So contextual BO does not search for one universal best co-catalyst loading.

Instead, it learns how the best loading changes with context:

$$
c \mapsto x^*(c).
$$

At each step, the context $c_t$ is supplied by the experimental situation, and BO chooses $x$ along the fixed-context slice:

$$
f(x,c_t).
$$

This can be summarised as:

> learn globally across $(x,c)$, but decide locally along the current context slice.

The final diagnostic showed that the model aligned best with the true contextual curves in the better-covered interior of the context range.

At the boundary contexts, such as $c=0.0$ and $c=1.0$, deviations were larger because the model had to extrapolate more strongly.

So contextual BO depends strongly on how well the context space is covered.

---

### Multi-fidelity vs contextual BO

The core distinction is:

- Fidelity controls evaluation accuracy and cost.
- Context controls the condition under which the design is judged.

In multi-fidelity BO:

- $s$ controls how accurate and expensive the evaluation is,
- low $s$ gives cheaper but biased information,
- and the final target remains $s=1$.

In contextual BO:

- $c$ controls the external condition,
- the model receives or observes that context,
- and the best design may genuinely differ across contexts.

So although both methods use augmented inputs, they answer different questions.

Multi-fidelity BO asks:

> **how accurately should we evaluate this design?**

Contextual BO asks:

> **under this condition, which design should we choose?**

---

### Final takeaway

This notebook extended BO in two practical directions.

The multi-fidelity section showed how BO can use cheap biased evaluations to guide the search toward a high-fidelity target under a limited budget.

The contextual section showed how BO can make recommendations that depend on external operating conditions rather than assuming that one design is best everywhere.

Together, they reinforce the broader message of Part 6:

> realistic BO is not only about optimising an objective function; it is about making structured decisions under constraints, uncertainty, cost, and context.

So the main takeaway is:

> **multi-fidelity BO changes how we evaluate candidates, while contextual BO changes how we interpret what “best” means under different conditions.**